## This .ipynb converts the raw datasets into point clouds (as .npy files)
## After downloading all the raw datasets into data/ folder, run this notebook
----

### For pre-training, it consists of all these files:
    data/IFC_extracted_elements_dataset_part_1_pointclouds_4096
    data/IFC_extracted_elements_dataset_part_2_pointclouds_4096
    data/IFC_extracted_elements_dataset_part_3_pointclouds_4096
    data/IFCNet_NO_TEST_pointclouds_4096
    data/processed_BIMGEOM_NO_TEST_pointclouds_4096

#### You should get all these 5 folders after running this notebook
----

### To install these packages to run the notebook, 
```
conda create -n [env_name] python=3.10
conda install conda-forge::trimesh==4.7.4 
pip install ifcopenshell==0.8.3.post2
conda install tqdm
conda activate [env_name]
```

### We used the following version (but higher versions are OK too):
```
ifcopenshell == 0.8.3.post2 
trimesh == 4.7.4
numpy >= 1.26.4
tqdm >= 4.67.1
```
----

In [ ]:
# import necessary libraries
import trimesh
import numpy as np
import os
from tqdm import tqdm
import shutil
import ifcopenshell # to convert ifc to obj
import ifcopenshell.geom

### Converting BIMGEOM and IFCNetCore into .npy files for point clouds:

In [ ]:
def convert_mesh_to_pointcloud(mesh_path, num_points=4096):
    """
    Loads a mesh file, samples a point cloud, normalizes it, and returns the points.
    
    Args:
        mesh_path (str): Path to the mesh file (.obj, .ply, etc.).
        num_points (int): The number of points to sample for the point cloud.
        
    Returns:
        np.ndarray: A numpy array of shape (num_points, 3) representing the
                    normalized point cloud, or None if the mesh fails to load.
    """
    try:
        # 1. Load the mesh using trimesh
        mesh = trimesh.load_mesh(mesh_path, process=False)

        # 2. Sample points uniformly from the mesh's surface
        # The 'trimesh.sample.sample_surface' function samples points
        # weighted by the area of each face, as described in the SpaRSE-BIM paper.
        points, _ = trimesh.sample.sample_surface(mesh, num_points)

        # 3. Normalize the point cloud to fit in a unit sphere
        # a. Center the point cloud at the origin
        points = points - points.mean(axis=0)
        # b. Scale the point cloud to fit within a sphere of radius 1
        points = points / np.max(np.linalg.norm(points, axis=1))

        return points.astype(np.float32)

    except Exception as e:
        print(f"Error processing {mesh_path}: {e}")
        return None

def process_dataset(input_dir, output_dir, num_points=4096):
    """
    Converts all .obj and .ply files in a directory to .npy point cloud files,
    preserving the subdirectory structure.
    """

    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Searching for mesh files in: {input_dir}")

    mesh_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".obj", ".ply")): # ply format is only applicable to BIMGEOM
                mesh_files.append(os.path.join(root, file))
    
    print(f"Found {len(mesh_files)} mesh files to convert.")


    for mesh_path in tqdm(mesh_files, desc="Converting meshes"):
        points = convert_mesh_to_pointcloud(mesh_path, num_points)
        
        if points is not None:
            # this preserves the folder structure (e.g., "IfcAirTerminal/train")
            relative_path = os.path.relpath(mesh_path, input_dir)
            output_filename = os.path.splitext(relative_path)[0] + '.npy'
            output_path = os.path.join(output_dir, output_filename)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            np.save(output_path, points)

if __name__ == '__main__':

    ifcnet_input_dir = 'data/IFCNetCore'
    ifcnet_output_dir = 'data/processed_IFCNetCore_pointclouds_4096'

    bimgeom_input_dir = 'data/BIMGEOM'
    bimgeom_output_dir = 'data/processed_BIMGEOM_pointclouds_4096'
    
    POINTS_PER_OBJECT = 4096

    print("--- Processing IFCNetCore Dataset ---")
    if os.path.isdir(ifcnet_input_dir):
        process_dataset(ifcnet_input_dir, ifcnet_output_dir, POINTS_PER_OBJECT)
    else:
        print(f"Directory not found: {ifcnet_input_dir}. Skipping.")

    print("\n--- Processing BIMGEOM Dataset ---")
    if os.path.isdir(bimgeom_input_dir):
        process_dataset(bimgeom_input_dir, bimgeom_output_dir, POINTS_PER_OBJECT)
    else:
        print(f"Directory not found: {bimgeom_input_dir}. Skipping.")
        
    print("\nConversion complete.")

### Converting IFC-884K to .npy files for point clouds:

#### a) There is an error with part 1 of IFC-884K. First, run the cell below to rename the wrong files

In [ ]:
BASE_PATH = "data"

FOLDERS_TO_SCAN = [
    "IFC_extracted_elements_dataset_part_1",
    "IFC_extracted_elements_dataset_part_2",
    "IFC_extracted_elements_dataset_part_3"
]

# the incorrect text to find and remove from filenames
INCORRECT_SUFFIX = ".ifc)"

# the target file extension the files should have
TARGET_EXTENSION = ".obj"


def fix_filenames_in_folders():
    """
    Scans specified folders and corrects filenames by removing an incorrect suffix.
    Example: '074508_IfcOpeningElement.ifc).obj' -> '074508_IfcOpeningElement.obj'
    """
    total_renamed_count = 0
    print("Starting filename cleanup script...")

    for folder_name in FOLDERS_TO_SCAN:
        target_directory = os.path.join(BASE_PATH, folder_name)

        if not os.path.isdir(target_directory):
            print(f"\nWARNING: Directory not found, skipping: {target_directory}")
            continue

        print(f"\nScanning folder: {folder_name}")
        renamed_in_folder = 0

        try:
            filenames = os.listdir(target_directory)
        except OSError as e:
            print(f"ERROR: Could not read directory. {e}")
            continue

        for filename in filenames:
            # Check if the filename ends with the specific incorrect pattern
            if filename.endswith(f"{INCORRECT_SUFFIX}{TARGET_EXTENSION}"):
                
                # Create the new, corrected filename by replacing the bad part
                new_filename = filename.replace(INCORRECT_SUFFIX, "")

                # Get the full old and new file paths
                old_filepath = os.path.join(target_directory, filename)
                new_filepath = os.path.join(target_directory, new_filename)

                try:
                    os.rename(old_filepath, new_filepath)
                    renamed_in_folder += 1
                except OSError as e:
                    print(f"ERROR renaming '{filename}': {e}")


        if renamed_in_folder > 0:
            print(f"Renamed {renamed_in_folder} file(s) in this folder.")
        else:
            print("No incorrectly named files found in this folder.")
        
        total_renamed_count += renamed_in_folder

    print("\n-----------------------------------------")
    print(f"Cleanup complete! Total files renamed: {total_renamed_count}")
    print("-----------------------------------------")


if __name__ == "__main__":
    fix_filenames_in_folders()

#### b) Converting to point clouds. Some samples are erroneous, but we simply ignore those as total samples are large.

In [ ]:
def convert_mesh_to_pointcloud(mesh_path, num_points=4096):
    """
    Loads a mesh file, samples a point cloud, normalizes it, and returns the points.
    
    Args:
        mesh_path (str): Path to the mesh file (.obj, .ply, etc.).
        num_points (int): The number of points to sample for the point cloud.
        
    Returns:
        np.ndarray: A numpy array of shape (num_points, 3) representing the
                    normalized point cloud, or None if the mesh fails to load.
    """
    try:
        # 1. Load the mesh using trimesh
        mesh = trimesh.load_mesh(mesh_path, process=False)

        # 2. Sample points uniformly from the mesh's surface
        # The 'trimesh.sample.sample_surface' function samples points
        # weighted by the area of each face, as described in the SpaRSE-BIM paper.
        points, _ = trimesh.sample.sample_surface(mesh, num_points)

        # 3. Normalize the point cloud to fit in a unit sphere
        # a. Center the point cloud at the origin
        points = points - points.mean(axis=0)
        # b. Scale the point cloud to fit within a sphere of radius 1
        points = points / np.max(np.linalg.norm(points, axis=1))

        return points.astype(np.float32)

    except Exception as e:
        print(f"Error processing {mesh_path}: {e}")
        return None

def process_dataset(input_dir, output_dir, num_points=4096):

    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Searching for mesh files in: {input_dir}")

    mesh_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".obj", ".ply")):
                mesh_files.append(os.path.join(root, file))
    
    print(f"Found {len(mesh_files)} mesh files to convert.")

    for mesh_path in tqdm(mesh_files, desc="Converting meshes"):
        points = convert_mesh_to_pointcloud(mesh_path, num_points)
        
        if points is not None:

            relative_path = os.path.relpath(mesh_path, input_dir)
            output_filename = os.path.splitext(relative_path)[0] + '.npy'
            output_path = os.path.join(output_dir, output_filename)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            np.save(output_path, points)

if __name__ == '__main__':

    POINTS_PER_OBJECT = 4096

    # loop through part numbers 1, 2, and 3 of the IFC-884K data
    for part in range(1, 4):

        input_directory = f'data/IFC_extracted_elements_dataset_part_{part}'
        output_directory = f'data/IFC_extracted_elements_dataset_part_{part}_pointclouds_4096'
        
        print(f"\n--- Processing Dataset: {os.path.basename(input_directory)} ---")
        
        if os.path.isdir(input_directory):
            process_dataset(input_directory, output_directory, POINTS_PER_OBJECT)
        else:
            print(f"Directory not found: {input_directory}. Skipping.")
            
    print("\nAll conversions complete.")

### Removing test set from IFCNetCore within IFCNet for pre-training:

In [ ]:
ifcnet_path = "data/IFCNet"
ifcnet_core_path = "data/IFCNetCore"
output_path = "data/IFCNet_NO_TEST" # used for pre-training without any test samples in downstream task
# ---------------------

def clean_ifcnet_dataset():
    """
    Creates a copy of IFCNet and removes files that are present
    in the 'test' subdirectories of IFCNetCore.
    """
    print("Starting the cleaning process...")

    # collect all test filenames from IFCNetCore
    test_filenames = set()
    print(f"\nScanning for test files in '{ifcnet_core_path}'...")

    for root, dirs, files in os.walk(ifcnet_core_path):
        # we are only interested in folders named 'test'
        if os.path.basename(root) == 'test':
            for filename in files:
                if filename.endswith(".obj"):
                    file_stem = os.path.splitext(filename)[0]
                    test_filenames.add(file_stem)

    if not test_filenames:
        print("Warning: No '.obj' test files found in IFCNetCore. Exiting.")
        return

    print(f"Found {len(test_filenames)} unique test filenames to exclude.")

    # create a fresh copy of the IFCNet directory
    print(f"\nCopying '{os.path.basename(ifcnet_path)}' to '{os.path.basename(output_path)}'...")

    if os.path.exists(output_path):
        print(f"   - Output folder already exists. Removing it first for a fresh copy.")
        shutil.rmtree(output_path)

    shutil.copytree(ifcnet_path, output_path)
    print("Copy complete.")

    # remove the test files from the new output directory
    files_removed_count = 0
    print(f"\nCleaning test files from '{os.path.basename(output_path)}'...")

    for root, dirs, files in os.walk(output_path):
        for filename in files:
            if filename.endswith(".ifc"):
                file_stem = os.path.splitext(filename)[0]

                if file_stem in test_filenames:
                    file_to_remove = os.path.join(root, filename)
                    os.remove(file_to_remove)
                    files_removed_count += 1

    print(f"Removed {files_removed_count} '.ifc' files.")
    print("\nProcess finished successfully! Your cleaned dataset is ready at:")
    print(output_path)


if __name__ == "__main__":
    clean_ifcnet_dataset()

### Converting IFCNet_NO_TEST to .obj format, then to .npy files

In [ ]:
ifc_source_dir = "data/IFCNet_NO_TEST"
obj_output_dir = "data/IFCNet_NO_TEST_obj"
# ---------------------


def convert_ifc_to_obj(ifc_path, obj_path):
    """
    Converts a single IFC file to an OBJ file.
    (This is the function you provided).
    """
    try:
        ifc_file = ifcopenshell.open(ifc_path)
        settings = ifcopenshell.geom.settings()
        settings.set(settings.USE_WORLD_COORDS, True)

        with open(obj_path, 'w') as obj_file:
            vertex_offset = 0
            products = ifc_file.by_type('IfcProduct')

            for product in products:
                if product.Representation is not None:
                    try:
                        shape = ifcopenshell.geom.create_shape(settings, product)
                        verts = shape.geometry.verts
                        faces = shape.geometry.faces

                        # Write Vertices
                        for i in range(0, len(verts), 3):
                            obj_file.write(f"v {verts[i]} {verts[i+1]} {verts[i+2]}\n")

                        # Write Faces
                        for i in range(0, len(faces), 3):
                            f1 = faces[i] + 1 + vertex_offset
                            f2 = faces[i+1] + 1 + vertex_offset
                            f3 = faces[i+2] + 1 + vertex_offset
                            obj_file.write(f"f {f1} {f2} {f3}\n")
                        
                        vertex_offset += len(verts) // 3
                    except Exception as e:
                        print(f"  - Warning: Could not process product {product.id()}: {e}")
    except Exception as e:
        print(f"  - Error: Failed to convert {os.path.basename(ifc_path)}. Reason: {e}")


def batch_convert_directory(input_dir, output_dir):
    """
    Finds all .ifc files in the input directory and its subdirectories,
    and converts them to .obj files in a mirrored output directory structure.
    """
    print("Starting batch conversion process...")
    
    # Safety check for the source directory
    if not os.path.isdir(input_dir):
        print(f"Error: Source directory not found at '{input_dir}'")
        return

    # Create the main output directory
    if os.path.exists(output_dir):
         print(f"ℹOutput directory '{output_dir}' already exists. Files will be overwritten if they exist.")
    os.makedirs(output_dir, exist_ok=True)

    total_files_found = 0
    converted_count = 0

    # Walk through the source directory tree
    for root, dirs, files in os.walk(input_dir):
        for filename in files:
            if filename.lower().endswith(".ifc"):
                total_files_found += 1
                
                # --- Construct input and output paths ---
                ifc_full_path = os.path.join(root, filename)

                # Determine the corresponding output subdirectory
                relative_path = os.path.relpath(root, input_dir)
                output_subdir = os.path.join(output_dir, relative_path)
                
                # Create the output subdirectory if it doesn't exist
                os.makedirs(output_subdir, exist_ok=True)
                
                # Create the final .obj filename
                obj_filename = os.path.splitext(filename)[0] + ".obj"
                obj_full_path = os.path.join(output_subdir, obj_filename)

                # --- Run the conversion ---
                print(f"\nProcessing [{total_files_found}] '{filename}'...")
                print(f"  -> Input:  {ifc_full_path}")
                print(f"  -> Output: {obj_full_path}")
                
                convert_ifc_to_obj(ifc_full_path, obj_full_path)
                converted_count += 1

    print("\n" + "="*50)
    print("Batch conversion complete!")
    print(f"Total .ifc files found: {total_files_found}")
    print(f"Files converted: {converted_count}")
    print(f"Your .obj files are located in: {output_dir}")
    print("="*50)


# --- Main execution ---
if __name__ == "__main__":
    # Ensure the IfcOpenShell library is installed
    try:
        import ifcopenshell
    except ImportError:
        print("Error: The 'ifcopenshell' library is required.")
        print("Please install it by running: pip install ifcopenshell")
        exit()

    batch_convert_directory(ifc_source_dir, obj_output_dir)

In [ ]:
def convert_mesh_to_pointcloud(mesh_path, num_points=4096):
    """
    Loads a mesh file, samples a point cloud, normalizes it, and returns the points.
    
    Args:
        mesh_path (str): Path to the mesh file (.obj, .ply, etc.).
        num_points (int): The number of points to sample for the point cloud.
        
    Returns:
        np.ndarray: A numpy array of shape (num_points, 3) representing the
                    normalized point cloud, or None if the mesh fails to load.
    """
    try:
        # 1. Load the mesh using trimesh
        mesh = trimesh.load_mesh(mesh_path, process=False)

        # 2. Sample points uniformly from the mesh's surface
        # The 'trimesh.sample.sample_surface' function samples points
        # weighted by the area of each face, as described in the SpaRSE-BIM paper.
        points, _ = trimesh.sample.sample_surface(mesh, num_points)

        # 3. Normalize the point cloud to fit in a unit sphere
        # a. Center the point cloud at the origin
        points = points - points.mean(axis=0)
        # b. Scale the point cloud to fit within a sphere of radius 1
        points = points / np.max(np.linalg.norm(points, axis=1))

        return points.astype(np.float32)

    except Exception as e:
        print(f"Error processing {mesh_path}: {e}")
        return None

def process_dataset(input_dir, output_dir, num_points=4096):
    """
    Converts all .obj and .ply files in a directory to .npy point cloud files,
    preserving the subdirectory structure.
    """

    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Searching for mesh files in: {input_dir}")

    mesh_files = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith((".obj", ".ply")): # ply format is only applicable to BIMGEOM
                mesh_files.append(os.path.join(root, file))
    
    print(f"Found {len(mesh_files)} mesh files to convert.")


    for mesh_path in tqdm(mesh_files, desc="Converting meshes"):
        points = convert_mesh_to_pointcloud(mesh_path, num_points)
        
        if points is not None:
            # this preserves the folder structure (e.g., "IfcAirTerminal/train")
            relative_path = os.path.relpath(mesh_path, input_dir)
            output_filename = os.path.splitext(relative_path)[0] + '.npy'
            output_path = os.path.join(output_dir, output_filename)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            np.save(output_path, points)

if __name__ == '__main__':

    ifcnet_input_dir = 'data/IFCNet_NO_TEST_obj'
    ifcnet_output_dir = 'data/processed_IFCNet_NO_TEST_pointclouds_4096'
    
    POINTS_PER_OBJECT = 4096

    print("--- Processing IFCNetCore Dataset ---")
    if os.path.isdir(ifcnet_input_dir):
        process_dataset(ifcnet_input_dir, ifcnet_output_dir, POINTS_PER_OBJECT)
    else:
        print(f"Directory not found: {ifcnet_input_dir}. Skipping.")

    print("\nConversion complete.")

### For BIMGEOM, we separate out the train and test folders. Train folders will be used for pre-training.

In [ ]:
def restructure_dataset(source_dir, dest_dir):
    """
    Copies .npy files from 'train' subdirectories of a source folder to a new
    destination folder, preserving the parent directory structure but omitting
    the 'train' and 'test' folders.

    Args:
        source_dir (str): The path to the root directory of the original dataset.
        dest_dir (str): The path to the new directory where the filtered
                        dataset will be saved.
    """

    print(f"Creating destination folder: {dest_dir}")
    os.makedirs(dest_dir, exist_ok=True)

    try:
        ifc_element_folders = [
            item for item in os.listdir(source_dir)
            if os.path.isdir(os.path.join(source_dir, item))
        ]
        if not ifc_element_folders:
            print(f"Warning: No subdirectories found in {source_dir}.")
            return
    except FileNotFoundError:
        print(f"Error: The source directory was not found: {source_dir}")
        return

    print(f"Found {len(ifc_element_folders)} IFC element folders to process.")


    for ifc_folder in tqdm(ifc_element_folders, desc="Processing IFC Folders"):

        train_folder_path = os.path.join(source_dir, ifc_folder, 'train')

        if os.path.isdir(train_folder_path):

            dest_ifc_path = os.path.join(dest_dir, ifc_folder)
            os.makedirs(dest_ifc_path, exist_ok=True)

            npy_files = [
                f for f in os.listdir(train_folder_path) if f.endswith('.npy')
            ]

            # copy each .npy file to the new destination
            for filename in npy_files:
                source_file = os.path.join(train_folder_path, filename)
                dest_file = os.path.join(dest_ifc_path, filename)

                shutil.copy2(source_file, dest_file)
        else:

            print(f"\nSkipping '{ifc_folder}': 'train' subfolder not found.")

if __name__ == '__main__':

    source_directory = 'data/processed_BIMGEOM_pointclouds_4096'

    output_directory_name = os.path.basename(source_directory).replace(
        'processed_BIMGEOM', 'processed_BIMGEOM_NO_TEST'
    )

    output_directory = os.path.join(os.path.dirname(source_directory), output_directory_name)

    print("--- Starting Dataset Restructuring ---")
    restructure_dataset(source_directory, output_directory)
    print("\nProcessing complete.")
    print(f"Your new dataset is located at: {output_directory}")